# Makola Market Retail & POS Optimization Case Study

### Overview
This project analyzes 120,000+ retail transactions from open-air market trading to uncover insights on profit margins, customer lifetime value, and inventory risks. The findings support migrating traditional vendors to digital tablet POS systems (Caris Retail Cloud) and provide data-backed risk scoring for financial partners evaluating merchant credit and stock loss insurance.

### Key Focus Areas
* **Sales & Margin Analysis:** Identifying top revenue drivers versus highest net-margin product categories.
* **Customer Behavior:** Tracking purchase frequency and customer value segments to optimize cross-selling.
* **Operational Risk:** Modeling vendor cash-flow velocity and predicting stockout vulnerabilities during peak market rushes.

In [ ]:
import pandas as pd

df = pd.read_csv('supermarket_large_dataset.csv')

df.head()

In [ ]:
df.info()

In [ ]:
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

## 1. Data Cleaning & Hygiene
Checking data structures, missing values, and formatting types to prepare the raw trade logs for analysis.

In [ ]:
df.isnull().sum()[df.isnull().sum() > 0].sort_values(ascending=False).head(10)

In [ ]:
missing_cols = ['complaint_category', 'refund_method', 'return_condition', 
                'return_reason', 'holiday_name', 'holiday_type', 
                'special_event', 'promotion_id', 'promotion_channel', 'card_type']

df[missing_cols] = df[missing_cols].fillna('None')

df[missing_cols].isnull().sum()

In [ ]:
print(df.columns.tolist())


In [ ]:
sales_by_category = df.groupby('product_category')[['sales_revenue', 'units_sold']].sum().reset_index()

sales_by_category = sales_by_category.sort_values(by='sales_revenue', ascending=False)

sales_by_category.head(10)

In [ ]:
store_sales = df.groupby('store_type')['sales_revenue'].sum().reset_index()

store_sales = store_sales.sort_values(by='sales_revenue', ascending=False)

store_sales

In [ ]:
segment_sales = df.groupby('customer_segment')['sales_revenue'].sum().reset_index()

segment_sales = segment_sales.sort_values(by='sales_revenue', ascending=False)

segment_sales

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.bar(sales_by_category['product_category'].head(5), sales_by_category['sales_revenue'].head(5))
plt.title('Top 5 Product Categories by Revenue')
plt.xlabel('Category')
plt.ylabel('Sales Revenue')
plt.show()

plt.figure(figsize=(10, 5))
plt.bar(segment_sales['customer_segment'].head(5), segment_sales['sales_revenue'].head(5))
plt.title('Top 5 Customer Segments by Revenue')
plt.xlabel('Customer Segment')
plt.ylabel('Sales Revenue')
plt.show()

## 2. Margin & Profitability Analysis
Evaluating revenue versus cost of goods sold (COGS) to discover which product categories drive the highest net profit margins.

In [ ]:
pro_profit_analysis = (
    df.groupby('product_category')
      .agg(
          total_revenue=('sales_revenue', 'sum'),
          total_cost=('cost_of_goods_sold', 'sum'),
          total_profit=('gross_profit', 'sum'),
          avg_margin=('profit_margin_percent', 'mean')
      )
      .reset_index()
      .sort_values(by='total_profit', ascending=False)
)

pro_profit_analysis.head(10)

## 4. Vendor Liquidity & Supply Chain Risk Modeling
Evaluating stockout rates, return rates, and inventory vulnerabilities to help market vendors optimize supply chains and assist financial partners with risk assessment.

In [ ]:
pro_supply_analysis = (
    df.groupby('product_category')
      .agg(
          avg_stockout_rate=('product_stockout_rate', 'mean'),
          avg_return_rate=('return_rate', 'mean'),
          avg_defect_rate=('defect_rate', 'mean')
      )
      .reset_index()
      .sort_values(by='avg_stockout_rate', ascending=False)
)

pro_supply_analysis.head(10)

## 3. Customer Lifetime Value & Segmentation
Analyzing customer segments by average lifetime value, engagement scores, and retention rates to uncover shopper loyalty trends and value drivers.

In [ ]:
pro_clv_analysis = (
    df.groupby('customer_segment')
      .agg(
          total_customers=('customer_id', 'nunique'),
          avg_clv=('customer_customer_lifetime_value', 'mean') if 'customer_customer_lifetime_value' in df.columns else ('customer_lifetime_value', 'mean'),
          avg_engagement=('customer_engagement_score', 'mean'),
          avg_retention=('customer_retention_score', 'mean')
      )
      .reset_index()
      .sort_values(by='avg_clv', ascending=False)
)

pro_clv_analysis

## 4. Product Affinity & Correlation Matrix
Analyzing statistical correlations between key operational variables (such as sales revenue, profit, units sold, and stockout rates) to understand cross-selling drivers and financial relationships.

In [ ]:
corr_features = ['sales_revenue', 'gross_profit', 'units_sold', 'discount_amount', 
                 'customer_engagement_score', 'customer_retention_score', 'product_stockout_rate']

corr_matrix = df[corr_features].corr()
corr_matrix

### Visualizing Feature Correlations
Plotting the correlation heatmap to visually verify relationships between revenue, profitability, units sold, and operational risk metrics.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.imshow(corr_matrix, cmap='coolwarm', aspect='auto', vmin=-1, vmax=1)
plt.colorbar()
plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=45, ha='right')
plt.yticks(range(len(corr_matrix.index)), corr_matrix.index)
plt.title('Feature Correlation Matrix - Supermarket Analytics')
plt.tight_layout()
plt.show()

## 6. Vendor Liquidity & Stockout Risk Modeling
Calculating cash conversion velocity and predictive stockout risk scores to evaluate inventory vulnerabilities and assist financial partners with alternative micro-merchant credit scoring.

In [ ]:
basket_matrix = pd.crosstab(df['customer_id'], df['product_category'])
basket_binary = basket_matrix.map(lambda x: 1 if x > 0 else 0)

category_corr = basket_binary.corr()
category_corr.head(10)

## 6. Vendor Liquidity & Stockout Risk Modeling
Calculating cash conversion velocity and predictive stockout risk scores to evaluate inventory vulnerabilities and assist financial partners with alternative micro-merchant credit scoring.

In [ ]:
df['cash_conversion_days'] = (df['cost_of_goods_sold'] / (df['sales_revenue'] + 1)) * 30

liquidity_analysis = df.groupby('store_type').agg(
    total_sales=('sales_revenue', 'sum'),
    avg_turnover_days=('cash_conversion_days', 'mean'),
    avg_profit_margin=('gross_margin_percentage', 'mean') if 'gross_margin_percentage' in df.columns else ('gross_profit', 'mean')
).reset_index()

liquidity_analysis.sort_values(by='avg_turnover_days', ascending=True)

## 6. Vendor Liquidity & Stockout Risk Modeling
Calculating cash conversion velocity and predictive stockout risk scores to evaluate inventory vulnerabilities and assist financial partners with alternative micro-merchant credit scoring.

In [ ]:
stockout_risk = df.groupby('product_category').agg(
    total_units_sold=('units_sold', 'sum'),
    avg_stock_level=('inventory_level', 'mean') if 'inventory_level' in df.columns else ('units_sold', 'mean'),
    return_rate=('return_rate', 'mean') if 'return_rate' in df.columns else ('units_sold', 'count')
).reset_index()

# Calculate a stockout risk index based on sales volume versus available stock
stockout_risk['stockout_risk_score'] = (stockout_risk['total_units_sold'] / (stockout_risk['avg_stock_level'] + 1)) * 100
stockout_risk.sort_values(by='stockout_risk_score', ascending=False).head(10)